# N-ATLAS Dev Kit — run a free N-ATLAS API on Google Colab

1. **Runtime → Change runtime type → T4 GPU**
2. Accept the model terms at https://huggingface.co/NCAIR1/N-ATLaS (log in first)
3. Get a free ngrok token at https://dashboard.ngrok.com/get-started/your-authtoken
4. Run every cell. The last cell prints a public URL you can give to any OpenAI-compatible app.

> N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.

In [ ]:
!git clone https://github.com/YOUR_GITHUB/natlas-devkit.git
%cd natlas-devkit/server
!CMAKE_ARGS="-DGGML_CUDA=on" pip install -q -r requirements.txt pyngrok

In [ ]:
import os
os.environ['NATLAS_BACKEND'] = 'llamacpp'          # quantized GGUF, fast on T4
os.environ['NATLAS_GPU_LAYERS'] = '-1'             # put every layer on the GPU
os.environ['NATLAS_GGUF_FILE'] = 'N-ATLaS-GGUF-Q4_K_M.gguf'
os.environ['NATLAS_API_KEY'] = 'change-me'          # testers send this as a Bearer token
NGROK_TOKEN = ''  # paste your ngrok token

In [ ]:
import subprocess, time, requests
proc = subprocess.Popen(['python', '-m', 'natlas_server'])
for _ in range(120):
    try:
        if requests.get('http://localhost:8000/health').ok: break
    except Exception: pass
    time.sleep(5)
print(requests.get('http://localhost:8000/health').json())

In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
url = ngrok.connect(8000).public_url
print('Base URL for SDKs / OpenAI clients:', url + '/v1')

In [ ]:
from openai import OpenAI  # proves OpenAI compatibility
client = OpenAI(base_url='http://localhost:8000/v1', api_key=os.environ['NATLAS_API_KEY'])
r = client.chat.completions.create(model='n-atlas', messages=[{'role':'user','content':'menene ake nufi da gwagwarmaya'}])
print(r.choices[0].message.content)

In [ ]:
print(requests.get('http://localhost:8000/v1/stats', headers={'Authorization': 'Bearer ' + os.environ['NATLAS_API_KEY']}).json())